# 03. Source-robust feature variants

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.training import transfer, tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
cohorts = list(transfer.FULL_COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
print("Development features:", features_path)

In [ ]:
spatial_selection_run = ""  # Optional pipeline_outputs/transfer/spatial_selection_<id>; blank uses the latest
spatial_selection_run = Path(spatial_selection_run) if spatial_selection_run else tuning.latest_run(root, "spatial_selection", area=transfer.AREA)
parameters = json.loads((spatial_selection_run / "selected.json").read_text())
print("spatial_selection run:", spatial_selection_run)
parameters

## Feature variants

In [ ]:
variants = list(transfer.VARIANTS)
fast = {cohort: {**params, "n_estimators": 300} for cohort, params in parameters.items()}
pd.DataFrame([{"variant": v, "predictors": transfer.matrix(tuning.cohort_split(development, cohorts[0])[0][:1], v).shape[1]}
              for v in variants])

## Spatial cross-validation and locked validation

In [ ]:
run = tuning.new_run(root, "features", features_path, area=transfer.AREA,
                     spatial_selection_run=str(spatial_selection_run), variants=variants)
records = []
for cohort in cohorts:
    train, validation = tuning.cohort_split(development, cohort)
    for variant in variants:
        pooled = transfer.spatial_cv(train, fast[cohort], variant)
        locked = transfer.locked_validation(train, validation, fast[cohort], variant)
        records.append({"cohort": cohort, "variant": variant, "spatial_macro_f1": pooled["macro_f1"],
                        "fold_min": min(pooled["fold_macro_f1"]), "fold_max": max(pooled["fold_macro_f1"]),
                        "locked_validation_macro_f1": locked["macro_f1"],
                        **{f"spatial_{crop}_f1": v["f1"] for crop, v in pooled["per_crop"].items()}})
        print(cohort, variant, f"spatial {pooled['macro_f1']:.3f}", flush=True)
tuning.save_records(run, "variant_scores", records)
display(pd.DataFrame(records).round(3))

## Leave-one-source-out by variant

In [ ]:
holdout = []
for cohort in cohorts:
    rows = transfer.development_rows(development, cohort)
    for variant in variants:
        for record in transfer.leave_one_source_out(rows, fast[cohort], variant):
            holdout.append({"cohort": cohort, **record})
tuning.save_records(run, "variant_leave_one_source_out", holdout)
display(pd.DataFrame([r for r in holdout if r["status"] == "scored"]).round(3))

## Source predictability by variant

In [ ]:
predictability = []
for cohort in cohorts:
    rows = transfer.development_rows(development, cohort)
    for variant in variants:
        predictability.extend({"cohort": cohort, **r} for r in transfer.source_predictability(rows, variant))
tuning.save_records(run, "variant_source_predictability", predictability)
display(pd.DataFrame([r for r in predictability if r["status"] == "scored"]).round(3))

## Variant summary

In [ ]:
holdout_mean = (pd.DataFrame([r for r in holdout if r["status"] == "scored"])
                .groupby(["cohort", "variant"]).macro_f1_scored_crops.mean().rename("source_holdout_macro_f1_mean"))
source_signal = (pd.DataFrame([r for r in predictability if r["status"] == "scored"])
                 .groupby(["cohort", "variant"]).source_balanced_accuracy.mean().rename("source_predictability_mean"))
summary = pd.DataFrame(records).set_index(["cohort", "variant"]).join([holdout_mean, source_signal]).reset_index()
summary.to_csv(run / "variant_summary.csv", index=False)
display(summary[["cohort", "variant", "spatial_macro_f1", "locked_validation_macro_f1",
                 "source_holdout_macro_f1_mean", "source_predictability_mean"]].round(3))

## Selected variant

In [ ]:
selected = {}
for cohort in cohorts:
    table = summary[summary.cohort == cohort]
    selected[cohort] = {"variant": table.sort_values("spatial_macro_f1", ascending=False).iloc[0].variant,
                        "series_variant": table[table.variant.isin(transfer.SERIES_VARIANTS)]
                        .sort_values("spatial_macro_f1", ascending=False).iloc[0].variant,
                        "params": parameters[cohort]}
(run / "selected.json").write_text(json.dumps(selected, indent=2))
selected

## Figure

In [ ]:
figure, axes = plt.subplots(1, len(cohorts), figsize=(7 * len(cohorts), 4), sharey=True)
for axis, cohort in zip(axes, cohorts):
    summary[summary.cohort == cohort].set_index("variant")[
        ["spatial_macro_f1", "locked_validation_macro_f1", "source_holdout_macro_f1_mean"]].plot(kind="bar", ax=axis)
    axis.set_title(cohort)
    axis.set_ylim(0, 1)
show_and_save(figure, run, "feature_variants")
print("Run folder:", run)